<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Heisenberg's lab
</font>
</h1>

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
The Dataset
</font>
</h2>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>

</font>
</p>


<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
In this section, <code>import</code> your required libraries and tools, and read the data files saved in the <code>Data</code> folder under the names <code>train.csv</code> and <code>test.csv</code>, and load them into your workspace.
</font>
</p>

In [31]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

<h2 dir="ltr" align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Preprocessing and Feature Engineering
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;In this question, you can use any preprocessing/feature engineering technique of your choice.
&nbsp;&nbsp;&nbsp;&nbsp;<br>
&nbsp;&nbsp;&nbsp;&nbsp;The techniques you use will <b>not</b> be directly evaluated by the judging system. Rather, they will all impact the accuracy of your model; therefore, the better preprocessing/feature engineering you perform to improve the model's accuracy, the more points you will earn from this question.
&nbsp;&nbsp;&nbsp;&nbsp;In this section, you can allocate a portion of the available data for validation.
</font>
</p>

In [32]:
targets = ['Marijuana', 'LSD', 'Mushroom', 'Psychotropic', 'Ex']

def build_psychedelic_features(train_d, test_d):
    tr = train_d.copy()
    te = test_d.copy()
    
    for df in [tr, te]:
        df['psychedelic_affinity'] = df['OpennessScore'] * 2.5 + df['NoveltySeekingScore'] - df['SelfDisciplineScore']
        df['unconventional_index'] = (df['NoveltySeekingScore'] + df['ImpulseControlScore']) - (df['CooperationScore'] + df['SelfDisciplineScore'])
        df['open_neuro'] = df['OpennessScore'] / (df['EmotionalStabilityScore'].abs() + 0.5)
        
    return tr, te

train_feat, test_feat = build_psychedelic_features(train_df, test_df)

features = [c for c in train_feat.columns if c not in targets + ['ID']]
X_train = train_feat[features].copy()
X_test = test_feat[features].copy()

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Model Training
</font>
</h2>



In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold

base_targets = ['Ex', 'Marijuana']
hard_targets = ['LSD', 'Mushroom', 'Psychotropic']

oof_probs_all = {}
test_probs_all = {}

for target in base_targets:
    y = train_feat[target].astype(int)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_p = np.zeros((len(train_feat), 3))
    test_p = np.zeros((len(test_feat), 3))
    
    for tr_idx, va_idx in skf.split(X_train, y):
        X_tr, y_tr = X_train.iloc[tr_idx], y.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        
        lgb = LGBMClassifier(objective='multiclass', num_class=3, class_weight='balanced',
                             n_estimators=300, learning_rate=0.03, max_depth=4, verbose=-1, random_state=42)
        lgb.fit(X_tr, y_tr)
        
        oof_p[va_idx] = lgb.predict_proba(X_va)
        test_p += lgb.predict_proba(X_test) / 5.0
        
    oof_probs_all[target] = oof_p
    test_probs_all[target] = test_p

X_tr_rich = X_train.copy()
X_te_rich = X_test.copy()

for t in base_targets:
    for c in range(3):
        X_tr_rich[f'{t}_prob_{c}'] = oof_probs_all[t][:, c]
        X_te_rich[f'{t}_prob_{c}'] = test_probs_all[t][:, c]

ordinal_probs = {}

for target in hard_targets:
    y = train_feat[target].astype(int)
    
    y_ge1 = (y >= 1).astype(int)
    y_ge2 = (y >= 2).astype(int)
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    p_ge1_oof = np.zeros(len(train_feat))
    p_ge2_oof = np.zeros(len(train_feat))
    p_ge1_test = np.zeros(len(test_feat))
    p_ge2_test = np.zeros(len(test_feat))
    
    for tr_idx, va_idx in skf.split(X_tr_rich, y):
        X_tr, X_va = X_tr_rich.iloc[tr_idx], X_tr_rich.iloc[va_idx]
        
        m_ge1 = LGBMClassifier(n_estimators=350, learning_rate=0.03, max_depth=4,
                               scale_pos_weight=1.5, subsample=0.8, colsample_bytree=0.8,
                               verbose=-1, random_state=42)
        m_ge1.fit(X_tr, y_ge1.iloc[tr_idx])
        p_ge1_oof[va_idx] = m_ge1.predict_proba(X_va)[:, 1]
        p_ge1_test += m_ge1.predict_proba(X_te_rich)[:, 1] / 5.0
        
        m_ge2 = LGBMClassifier(n_estimators=250, learning_rate=0.03, max_depth=3,
                               scale_pos_weight=3.5, subsample=0.8, colsample_bytree=0.8,
                               verbose=-1, random_state=42)
        m_ge2.fit(X_tr, y_ge2.iloc[tr_idx])
        p_ge2_oof[va_idx] = m_ge2.predict_proba(X_va)[:, 1]
        p_ge2_test += m_ge2.predict_proba(X_te_rich)[:, 1] / 5.0
        
    ordinal_probs[target] = {
        'oof_ge1': p_ge1_oof,
        'oof_ge2': p_ge2_oof,
        'test_ge1': p_ge1_test,
        'test_ge2': p_ge2_test
    }

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Evaluation Metric
</font>
</h2>



In [ ]:
from sklearn.metrics import f1_score

final_predictions = {}
test_predictions = {}
total_competition_score = 0.0

for target in base_targets:
    y_true = train_feat[target].astype(int)
    p = oof_probs_all[target]
    pred_oof = np.argmax(p, axis=1)
    
    macro_f1 = f1_score(y_true, pred_oof, average='macro')
    score = 20.0 * min(1.0, max(0.0, (macro_f1 - 0.50) / 0.50)) if macro_f1 >= 0.66 else 0.0
    total_competition_score += score
    
    test_predictions[target] = np.argmax(test_probs_all[target], axis=1)
    print(f"Target: {target:15s} | Macro F1: {macro_f1:.4f} | Score: {score:.2f}/20")

for target in hard_targets:
    y_true = train_feat[target].astype(int)
    p1 = ordinal_probs[target]['oof_ge1']
    p2 = ordinal_probs[target]['oof_ge2']
    
    best_f1 = 0
    best_th = (0.5, 0.5)
    
    for th1 in np.linspace(0.25, 0.60, 35):
        for th2 in np.linspace(0.20, 0.70, 35):
            preds = np.zeros(len(y_true), dtype=int)
            preds[p1 >= th1] = 1
            preds[p2 >= th2] = 2
            
            sc = f1_score(y_true, preds, average='macro')
            if sc > best_f1:
                best_f1 = sc
                best_th = (th1, th2)
                
    th1, th2 = best_th
    pred_oof = np.zeros(len(y_true), dtype=int)
    pred_oof[p1 >= th1] = 1
    pred_oof[p2 >= th2] = 2
    
    t_p1 = ordinal_probs[target]['test_ge1']
    t_p2 = ordinal_probs[target]['test_ge2']
    pred_test = np.zeros(len(test_feat), dtype=int)
    pred_test[t_p1 >= th1] = 1
    pred_test[t_p2 >= th2] = 2
    test_predictions[target] = pred_test
    
    score = 20.0 * min(1.0, max(0.0, (best_f1 - 0.50) / 0.50)) if best_f1 >= 0.66 else 0.0
    total_competition_score += score
    print(f"Target: {target:15s} | Macro F1: {best_f1:.4f} | Thresh: (th1={th1:.2f}, th2={th2:.2f}) | Score: {score:.2f}/20")

print(f"\nFinal Calculated Competition Score: {total_competition_score:.2f} / 100")

Target: Ex              | Macro F1: 0.9803 | Score: 19.21/20
Target: Marijuana       | Macro F1: 0.9378 | Score: 17.51/20
Target: LSD             | Macro F1: 0.5580 | Thresh: (th1=0.50, th2=0.41) | Score: 0.00/20
Target: Mushroom        | Macro F1: 0.5290 | Thresh: (th1=0.46, th2=0.20) | Score: 0.00/20
Target: Psychotropic    | Macro F1: 0.5479 | Thresh: (th1=0.44, th2=0.44) | Score: 0.00/20

Final Calculated Competition Score: 36.72 / 100


<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Prediction on Test Data
</font>
</h2>


In [38]:
submission = pd.DataFrame({'ID': test_df['ID']})
for target in targets:
    submission[target] = test_predictions[target]

submission.head()

,ID,Marijuana,LSD,Mushroom,Psychotropic,Ex
0,1,0,0,0,0,0
1,2,0,0,0,0,0
2,3,0,0,1,1,2
3,4,0,0,0,0,0
4,5,0,0,0,0,0


<h2 dir="ltr" align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>Result Generator Cell</b>
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
&nbsp;&nbsp;&nbsp;&nbsp;Run the cell below to generate the <code>result.zip</code> file. Please note that you must save the changes made in the notebook (<code>ctrl+s</code>) before running the cell below; otherwise, your score will be changed to zero at the end of the competition.
&nbsp;&nbsp;&nbsp;&nbsp;<br>
&nbsp;&nbsp;&nbsp;&nbsp;Also, if you are using Colab to run this notebook file, download the latest version of your notebook and place it inside the submission file before submitting the <code>result.zip</code> file.
</font>
</p>

In [39]:
import zipfile
import joblib
import os

if not os.path.exists(os.path.join(os.getcwd(), 'Drug.ipynb')):
    %notebook -e Drug.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['Drug.ipynb', 'submission.csv']
compress(file_names)

File Paths:
['Drug.ipynb', 'submission.csv']
